In [ ]:
import importlib.util
import os
import platform
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import torch


def module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (AttributeError, ImportError, ModuleNotFoundError, ValueError):
        return False


def absolute_root(value, setting_name):
    root = Path(value)

    if not root.is_absolute():
        raise ValueError(
            f"{setting_name} deve indicare un path assoluto: {root}"
        )

    return root


IN_COLAB = module_available("google.colab")
PROJECT_ROOT_OVERRIDE = os.environ.get("GRAVITYSPY_PROJECT_ROOT")
LOCAL_RUNTIME_ROOT_OVERRIDE = os.environ.get(
    "GRAVITYSPY_LOCAL_RUNTIME_ROOT"
)
PROJECT_ROOT_IS_EXPLICIT = PROJECT_ROOT_OVERRIDE is not None
LOCAL_RUNTIME_ROOT_IS_EXPLICIT = LOCAL_RUNTIME_ROOT_OVERRIDE is not None

COLAB_DRIVE_MOUNT_POINT = Path("/content/drive")
COLAB_MYDRIVE_ROOT = COLAB_DRIVE_MOUNT_POINT / "MyDrive"
DEFAULT_COLAB_PROJECT_ROOT = COLAB_MYDRIVE_ROOT / "TESI_SAKAI2022"

if PROJECT_ROOT_IS_EXPLICIT:
    PROJECT_ROOT = absolute_root(
        PROJECT_ROOT_OVERRIDE,
        "GRAVITYSPY_PROJECT_ROOT",
    )
elif IN_COLAB:
    PROJECT_ROOT = DEFAULT_COLAB_PROJECT_ROOT
else:
    raise RuntimeError(
        "Fuori da Colab impostare GRAVITYSPY_PROJECT_ROOT "
        "su una root persistente assoluta."
    )

if LOCAL_RUNTIME_ROOT_IS_EXPLICIT:
    LOCAL_RUNTIME_ROOT = absolute_root(
        LOCAL_RUNTIME_ROOT_OVERRIDE,
        "GRAVITYSPY_LOCAL_RUNTIME_ROOT",
    )
elif IN_COLAB:
    LOCAL_RUNTIME_ROOT = Path("/content")
else:
    LOCAL_RUNTIME_ROOT = absolute_root(
        tempfile.gettempdir(),
        "tempfile.gettempdir()",
    ) / "gravityspy_runtime"


def to_gib(byte_count):
    return byte_count / (1024 ** 3)


def read_total_ram_bytes():
    meminfo = Path("/proc/meminfo")

    if meminfo.is_file():
        with meminfo.open("r", encoding="utf-8") as file:
            first_line = file.readline()

        total_kib = int(first_line.split()[1])
        return total_kib * 1024

    if hasattr(os, "sysconf"):
        try:
            return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
        except (KeyError, OSError, TypeError, ValueError):
            pass

    return None


def existing_disk_probe(path):
    probe = path

    while not probe.exists():
        parent = probe.parent

        if parent == probe:
            raise RuntimeError(
                f"Nessun percorso esistente per inventario disco: {path}"
            )

        probe = parent

    return probe


print("=== ROOT CONTRACT ===")
print("IN_COLAB =", IN_COLAB)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("LOCAL_RUNTIME_ROOT =", LOCAL_RUNTIME_ROOT)

print("\n=== SOFTWARE ===")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA PyTorch:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())

print("\n=== GPU ===")
print("CUDA disponibile:", torch.cuda.is_available())

if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)

    print("Nome GPU:", torch.cuda.get_device_name(0))
    print("VRAM totale GiB:", round(to_gib(properties.total_memory), 2))

    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,driver_version,memory.total",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    print("nvidia-smi:", result.stdout.strip())

print("\n=== RAM ===")
ram_total_bytes = read_total_ram_bytes()

if ram_total_bytes is None:
    print("RAM totale GiB: non disponibile")
else:
    print("RAM totale GiB:", round(to_gib(ram_total_bytes), 2))

print("\n=== DISCO LOCAL_RUNTIME_ROOT ===")
disk_probe = existing_disk_probe(LOCAL_RUNTIME_ROOT)
disk = shutil.disk_usage(disk_probe)
print("Percorso analizzato:", disk_probe)
print("Totale GiB:", round(to_gib(disk.total), 2))
print("Usato GiB:", round(to_gib(disk.used), 2))
print("Libero GiB:", round(to_gib(disk.free), 2))


=== SOFTWARE ===
Python: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch: 2.11.0+cu128
CUDA PyTorch: 12.8
cuDNN: 91900

=== GPU ===
CUDA disponibile: True
Nome GPU: Tesla T4
VRAM totale GiB: 14.56
nvidia-smi: Tesla T4, 580.82.07, 15360 MiB

=== RAM ===
RAM totale GiB: 12.67

=== DISCO /content ===
Totale GiB: 112.64
Usato GiB: 47.01
Libero GiB: 65.61


In [1]:
if IN_COLAB and not PROJECT_ROOT_IS_EXPLICIT:
    from google.colab import drive

    if not COLAB_MYDRIVE_ROOT.is_dir():
        drive.mount(str(COLAB_DRIVE_MOUNT_POINT), force_remount=False)
    else:
        print("Drive già disponibile:", COLAB_DRIVE_MOUNT_POINT)
else:
    print("Mount Drive non richiesto; PROJECT_ROOT =", PROJECT_ROOT)


Mounted at /content/drive


In [2]:
from pathlib import Path
import hashlib


TRANSFER_DIR = PROJECT_ROOT / "sakai224_full" / "transfer_v1"

EXPECTED = {
    "sakai224_full_out272_gray.tar": {
        "bytes": 1171223040,
        "sha256": "70c766c3652f586c8da4c8cadf134a8ce83cfd07c8cd10545367f380017ddab2",
    },
    "sakai224_full_central224.tar": {
        "bytes": 1727038464,
        "sha256": "0fad37cdf4ea056e2c3abfa2d1be73b9d9bd55e644d54a58013950697cf4d4b6",
    },
    "manifest_events_272.csv": {
        "bytes": 1298437,
        "sha256": "620960e6b35da31bc4cf1a8a6da8679027d378e1d4b2ac73c5267c1c5311ae82",
    },
    "manifest_tensors_224_central.csv": {
        "bytes": 1109634,
        "sha256": "94b7e2e051bd32de72dd2cb6a5fb7eda9bcaec123cbf3a07d0d901bd19779afa",
    },
    "split_train_indices_seed123.csv": {
        "bytes": 567069,
        "sha256": "8c7a513b98dbe996261bc80965500ab15ab77f498ec9440858003152e42f926f",
    },
    "split_val_indices_seed123.csv": {
        "bytes": 149608,
        "sha256": "e63109c88df7b3ee97317f94791f26df3fcaf8b937921cfb27967d01c2546467",
    },
    "transfer_manifest_sha256.txt": {
        "bytes": 711,
        "sha256": "59a04c1098ec4465c1a9864e7bc61181c7981e316c8bc78adbf4c18b78da962f",
    },
}


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


print("TRANSFER_DIR =", TRANSFER_DIR)
print("DIRECTORY_EXISTS =", TRANSFER_DIR.is_dir())
print()

all_ok = True

for name, expected in EXPECTED.items():
    path = TRANSFER_DIR / name

    if not path.is_file():
        print(name)
        print("  EXISTS   = False ❌")
        all_ok = False
        continue

    actual_bytes = path.stat().st_size
    actual_sha256 = sha256_file(path)

    bytes_ok = actual_bytes == expected["bytes"]
    hash_ok = actual_sha256 == expected["sha256"]

    print(name)
    print("  EXISTS   = True")
    print("  BYTES    =", actual_bytes)
    print("  BYTES_OK =", bytes_ok)
    print("  SHA256   =", actual_sha256)
    print("  HASH_OK  =", hash_ok)
    print()

    if not (bytes_ok and hash_ok):
        all_ok = False


print("ALL_FILES_OK =", all_ok)

TRANSFER_DIR = /content/drive/MyDrive/TESI_SAKAI2022/sakai224_full/transfer_v1
DIRECTORY_EXISTS = True

sakai224_full_out272_gray.tar
  EXISTS   = True
  BYTES    = 1171223040
  BYTES_OK = True
  SHA256   = 70c766c3652f586c8da4c8cadf134a8ce83cfd07c8cd10545367f380017ddab2
  HASH_OK  = True

sakai224_full_central224.tar
  EXISTS   = True
  BYTES    = 1727038464
  BYTES_OK = True
  SHA256   = 0fad37cdf4ea056e2c3abfa2d1be73b9d9bd55e644d54a58013950697cf4d4b6
  HASH_OK  = True

manifest_events_272.csv
  EXISTS   = True
  BYTES    = 1298437
  BYTES_OK = True
  SHA256   = 620960e6b35da31bc4cf1a8a6da8679027d378e1d4b2ac73c5267c1c5311ae82
  HASH_OK  = True

manifest_tensors_224_central.csv
  EXISTS   = True
  BYTES    = 1109634
  BYTES_OK = True
  SHA256   = 94b7e2e051bd32de72dd2cb6a5fb7eda9bcaec123cbf3a07d0d901bd19779afa
  HASH_OK  = True

split_train_indices_seed123.csv
  EXISTS   = True
  BYTES    = 567069
  BYTES_OK = True
  SHA256   = 8c7a513b98dbe996261bc80965500ab15ab77f498ec9440858003152e

In [3]:
from pathlib import Path
import shutil

LOCAL_TRANSFER_DIR = LOCAL_RUNTIME_ROOT / "sakai224_full_transfer_v1"

if LOCAL_TRANSFER_DIR.exists():
    raise RuntimeError(
        f"STOP: la cartella locale esiste già: {LOCAL_TRANSFER_DIR}"
    )

LOCAL_TRANSFER_DIR.mkdir(parents=True)

print("Copia dei 7 file dallo storage persistente al runtime locale...")

for name in EXPECTED:
    source = TRANSFER_DIR / name
    destination = LOCAL_TRANSFER_DIR / name

    print("Copio:", name)
    shutil.copy2(source, destination)

print()
print("LOCAL_COPY_COMPLETE = True")
print("LOCAL_TRANSFER_DIR =", LOCAL_TRANSFER_DIR)

Copia dei 7 file da Drive a /content...
Copio: sakai224_full_out272_gray.tar
Copio: sakai224_full_central224.tar
Copio: manifest_events_272.csv
Copio: manifest_tensors_224_central.csv
Copio: split_train_indices_seed123.csv
Copio: split_val_indices_seed123.csv
Copio: transfer_manifest_sha256.txt

LOCAL_COPY_COMPLETE = True
LOCAL_TRANSFER_DIR = /content/sakai224_full_transfer_v1


In [4]:
print("=== VERIFICA COPIA LOCALE NEL RUNTIME ===\n")

local_all_ok = True

for name, expected in EXPECTED.items():
    path = LOCAL_TRANSFER_DIR / name

    if not path.is_file():
        print(name)
        print("  EXISTS   = False")
        local_all_ok = False
        continue

    actual_bytes = path.stat().st_size
    actual_sha256 = sha256_file(path)

    bytes_ok = actual_bytes == expected["bytes"]
    hash_ok = actual_sha256 == expected["sha256"]

    print(name)
    print("  BYTES_OK =", bytes_ok)
    print("  HASH_OK  =", hash_ok)
    print()

    if not (bytes_ok and hash_ok):
        local_all_ok = False

print("LOCAL_ALL_FILES_OK =", local_all_ok)

=== VERIFICA COPIA LOCALE /content ===

sakai224_full_out272_gray.tar
  BYTES_OK = True
  HASH_OK  = True

sakai224_full_central224.tar
  BYTES_OK = True
  HASH_OK  = True

manifest_events_272.csv
  BYTES_OK = True
  HASH_OK  = True

manifest_tensors_224_central.csv
  BYTES_OK = True
  HASH_OK  = True

split_train_indices_seed123.csv
  BYTES_OK = True
  HASH_OK  = True

split_val_indices_seed123.csv
  BYTES_OK = True
  HASH_OK  = True

transfer_manifest_sha256.txt
  BYTES_OK = True
  HASH_OK  = True

LOCAL_ALL_FILES_OK = True


In [5]:
from pathlib import Path
import subprocess

DATA_DIR = LOCAL_RUNTIME_ROOT / "sakai224_full_data"
OUT272_DIR = DATA_DIR / "out272_gray"

if OUT272_DIR.exists():
    raise RuntimeError(
        f"STOP: la directory esiste già: {OUT272_DIR}"
    )

DATA_DIR.mkdir(parents=True, exist_ok=True)

archive = LOCAL_TRANSFER_DIR / "sakai224_full_out272_gray.tar"

print("Estraggo:", archive.name)

subprocess.run(
    [
        "tar",
        "-xf",
        str(archive),
        "-C",
        str(DATA_DIR),
    ],
    check=True,
)

print("EXTRACTION_COMPLETE = True")

files = [
    path
    for path in OUT272_DIR.rglob("*")
    if path.is_file()
]

total_bytes = sum(path.stat().st_size for path in files)

print("Directory:", OUT272_DIR)
print("File:", len(files))
print("Expected files:", 34332)
print("FILE_COUNT_OK =", len(files) == 34332)

print("Bytes:", total_bytes)
print("Expected bytes:", 1144867540)
print("BYTES_OK =", total_bytes == 1144867540)

Estraggo: sakai224_full_out272_gray.tar
EXTRACTION_COMPLETE = True
Directory: /content/sakai224_full_data/out272_gray
File: 34332
Expected files: 34332
FILE_COUNT_OK = True
Bytes: 1144867540
Expected bytes: 1144867540
BYTES_OK = True


In [6]:
CENTRAL224_DIR = DATA_DIR / "tensors_224_central_uint8"

if CENTRAL224_DIR.exists():
    raise RuntimeError(
        f"STOP: la directory esiste già: {CENTRAL224_DIR}"
    )

archive = LOCAL_TRANSFER_DIR / "sakai224_full_central224.tar"

print("Estraggo:", archive.name)

subprocess.run(
    [
        "tar",
        "-xf",
        str(archive),
        "-C",
        str(DATA_DIR),
    ],
    check=True,
)

print("EXTRACTION_COMPLETE = True")

files = [
    path
    for path in CENTRAL224_DIR.rglob("*")
    if path.is_file()
]

total_bytes = sum(path.stat().st_size for path in files)

print("Directory:", CENTRAL224_DIR)
print("File:", len(files))
print("Expected files:", 8583)
print("FILE_COUNT_OK =", len(files) == 8583)

print("Bytes:", total_bytes)
print("Expected bytes:", 1722642432)
print("BYTES_OK =", total_bytes == 1722642432)


Estraggo: sakai224_full_central224.tar
EXTRACTION_COMPLETE = True
Directory: /content/sakai224_full_data/tensors_224_central_uint8
File: 8583
Expected files: 8583
FILE_COUNT_OK = True
Bytes: 1722642432
Expected bytes: 1722642432
BYTES_OK = True


In [7]:
import csv

METADATA_DIR = LOCAL_TRANSFER_DIR

CSV_EXPECTED_ROWS = {
    "manifest_events_272.csv": 8583,
    "manifest_tensors_224_central.csv": 8583,
    "split_train_indices_seed123.csv": 6866,
    "split_val_indices_seed123.csv": 1717,
}

print("=== CONTROLLO CSV LOCALI ===")

all_rows_ok = True

for name, expected_rows in CSV_EXPECTED_ROWS.items():
    path = METADATA_DIR / name

    with path.open("r", encoding="utf-8", newline="") as file:
        reader = csv.reader(file)
        header = next(reader)
        rows = sum(1 for _ in reader)

    rows_ok = rows == expected_rows

    print()
    print(name)
    print("  HEADER      =", header)
    print("  ROWS        =", rows)
    print("  EXPECTED    =", expected_rows)
    print("  ROWS_OK     =", rows_ok)

    if not rows_ok:
        all_rows_ok = False

print()
print("ALL_CSV_ROWS_OK =", all_rows_ok)

=== CONTROLLO CSV LOCALI ===

manifest_events_272.csv
  HEADER      = ['event_index', 'label', 'event_id', 'img_0_5_272', 'img_1_0_272', 'img_2_0_272', 'img_4_0_272']
  ROWS        = 8583
  EXPECTED    = 8583
  ROWS_OK     = True

manifest_tensors_224_central.csv
  HEADER      = ['event_index', 'label', 'event_id', 'tensor_path', 'channels', 'height', 'width', 'dtype', 'layout', 'channel_order', 'crop_left']
  ROWS        = 8583
  EXPECTED    = 8583
  ROWS_OK     = True

split_train_indices_seed123.csv
  HEADER      = ['split_name', 'split_pos', 'manifest_index', 'event_index', 'event_id', 'tensor_path']
  ROWS        = 6866
  EXPECTED    = 6866
  ROWS_OK     = True

split_val_indices_seed123.csv
  HEADER      = ['split_name', 'split_pos', 'manifest_index', 'event_index', 'event_id', 'tensor_path']
  ROWS        = 1717
  EXPECTED    = 1717
  ROWS_OK     = True

ALL_CSV_ROWS_OK = True


In [8]:
import csv

def load_split_csv(path):
    with path.open("r", encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


train_rows = load_split_csv(
    METADATA_DIR / "split_train_indices_seed123.csv"
)

val_rows = load_split_csv(
    METADATA_DIR / "split_val_indices_seed123.csv"
)


train_manifest_indices = {
    int(row["manifest_index"])
    for row in train_rows
}

val_manifest_indices = {
    int(row["manifest_index"])
    for row in val_rows
}

train_event_ids = {
    row["event_id"]
    for row in train_rows
}

val_event_ids = {
    row["event_id"]
    for row in val_rows
}


manifest_overlap = (
    train_manifest_indices
    & val_manifest_indices
)

event_id_overlap = (
    train_event_ids
    & val_event_ids
)

manifest_union = (
    train_manifest_indices
    | val_manifest_indices
)

expected_manifest_indices = set(range(8583))


print("=== CONTROLLO SPLIT ===")

print("TRAIN_ROWS =", len(train_rows))
print("VAL_ROWS   =", len(val_rows))

print()
print(
    "TRAIN_MANIFEST_INDEX_UNIQUE =",
    len(train_manifest_indices) == 6866,
)

print(
    "VAL_MANIFEST_INDEX_UNIQUE =",
    len(val_manifest_indices) == 1717,
)

print(
    "TRAIN_EVENT_ID_UNIQUE =",
    len(train_event_ids) == 6866,
)

print(
    "VAL_EVENT_ID_UNIQUE =",
    len(val_event_ids) == 1717,
)

print()
print(
    "MANIFEST_INDEX_OVERLAP =",
    len(manifest_overlap),
)

print(
    "EVENT_ID_OVERLAP =",
    len(event_id_overlap),
)

print()
print(
    "UNION_MANIFEST_INDEX_COUNT =",
    len(manifest_union),
)

print(
    "FULL_COVERAGE_8583 =",
    manifest_union == expected_manifest_indices,
)

print()
print(
    "EVENT_INDEX_RELATION_OK =",
    all(
        int(row["event_index"])
        == int(row["manifest_index"]) + 1
        for row in train_rows + val_rows
    ),
)

print(
    "SPLIT_POS_TRAIN_OK =",
    [int(row["split_pos"]) for row in train_rows]
    == list(range(6866)),
)

print(
    "SPLIT_POS_VAL_OK =",
    [int(row["split_pos"]) for row in val_rows]
    == list(range(1717)),
)


=== CONTROLLO SPLIT ===
TRAIN_ROWS = 6866
VAL_ROWS   = 1717

TRAIN_MANIFEST_INDEX_UNIQUE = True
VAL_MANIFEST_INDEX_UNIQUE = True
TRAIN_EVENT_ID_UNIQUE = True
VAL_EVENT_ID_UNIQUE = True

MANIFEST_INDEX_OVERLAP = 0
EVENT_ID_OVERLAP = 0

UNION_MANIFEST_INDEX_COUNT = 8583
FULL_COVERAGE_8583 = True

EVENT_INDEX_RELATION_OK = True
SPLIT_POS_TRAIN_OK = True
SPLIT_POS_VAL_OK = True


In [9]:
import csv

# Lettura dei tre file richiesti.
with (METADATA_DIR / "manifest_events_272.csv").open(
    "r", encoding="utf-8", newline=""
) as f:
    event_rows = list(csv.DictReader(f))

with (METADATA_DIR / "manifest_tensors_224_central.csv").open(
    "r", encoding="utf-8", newline=""
) as f:
    tensor_rows = list(csv.DictReader(f))

with (METADATA_DIR / "split_val_indices_seed123.csv").open(
    "r", encoding="utf-8", newline=""
) as f:
    val_rows = list(csv.DictReader(f))


# Selezione del primo evento della validation ufficiale.
val_row = val_rows[0]
manifest_index = int(val_row["manifest_index"])

event_row = event_rows[manifest_index]
tensor_row = tensor_rows[manifest_index]


print("=== EVENTO DI CONTROLLO ===")
print("manifest_index =", manifest_index)
print("event_index    =", event_row["event_index"])
print("event_id       =", event_row["event_id"])

print("\n=== QUATTRO VISTE 272x224 ===")
print("0.5 s =", event_row["img_0_5_272"])
print("1.0 s =", event_row["img_1_0_272"])
print("2.0 s =", event_row["img_2_0_272"])
print("4.0 s =", event_row["img_4_0_272"])

print("\n=== TENSORE CENTRALE ===")
print("tensor_path   =", tensor_row["tensor_path"])
print("channels      =", tensor_row["channels"])
print("height        =", tensor_row["height"])
print("width         =", tensor_row["width"])
print("dtype         =", tensor_row["dtype"])
print("layout        =", tensor_row["layout"])
print("channel_order =", tensor_row["channel_order"])
print("crop_left     =", tensor_row["crop_left"])

print("\n=== ALLINEAMENTO ===")
print(
    "EVENT_ID_MATCH =",
    event_row["event_id"] == tensor_row["event_id"] == val_row["event_id"],
)

=== EVENTO DI CONTROLLO ===
manifest_index = 6864
event_index    = 6865
event_id       = H1_4D7IcowE8a

=== QUATTRO VISTE 272x224 ===
0.5 s = out272_gray/img_027457.png
1.0 s = out272_gray/img_027458.png
2.0 s = out272_gray/img_027459.png
4.0 s = out272_gray/img_027460.png

=== TENSORE CENTRALE ===
tensor_path   = tensors_224_central_uint8/event_006865.bin
channels      = 4
height        = 224
width         = 224
dtype         = uint8
layout        = channel_major
channel_order = 0.5;1.0;2.0;4.0
crop_left     = 24

=== ALLINEAMENTO ===
EVENT_ID_MATCH = True


In [10]:
from pathlib import Path
from PIL import Image
import numpy as np


view_columns = [
    "img_0_5_272",
    "img_1_0_272",
    "img_2_0_272",
    "img_4_0_272",
]

png_paths = [
    DATA_DIR / event_row[column]
    for column in view_columns
]

tensor_path = DATA_DIR / tensor_row["tensor_path"]


print("=== ESISTENZA FILE ===")
print("ALL_PNG_PATHS_EXIST =", all(path.is_file() for path in png_paths))
print("TENSOR_PATH_EXISTS  =", tensor_path.is_file())


# Lettura delle quattro PNG senza modifiche.
png_arrays = []

print("\n=== PNG ORIGINALI ===")

for i, path in enumerate(png_paths):
    with Image.open(path) as image:
        array = np.asarray(image)

        print(
            f"channel {i}:",
            "mode =", image.mode,
            "size =", image.size,
            "shape =", array.shape,
            "dtype =", array.dtype,
        )

        png_arrays.append(array.copy())


all_png_shapes_ok = all(
    array.shape == (224, 272)
    for array in png_arrays
)

all_png_dtypes_ok = all(
    array.dtype == np.uint8
    for array in png_arrays
)


# Lettura del tensore centrale uint8 channel-major.
central = np.fromfile(
    tensor_path,
    dtype=np.uint8,
).reshape(4, 224, 224)


print("\n=== TENSORE CENTRALE ===")
print("shape =", central.shape)
print("dtype =", central.dtype)


# Ricostruzione del crop centrale direttamente dalle PNG 272x224.
crop_left = 24
crop_right = crop_left + 224

central_from_png = np.stack(
    [
        array[:, crop_left:crop_right]
        for array in png_arrays
    ],
    axis=0,
)


difference = (
    central_from_png.astype(np.int16)
    - central.astype(np.int16)
)

diff_count = np.count_nonzero(difference)
max_abs_diff = np.abs(difference).max()


print("\n=== CONTROLLI ===")
print("ALL_PNG_SHAPES_OK =", all_png_shapes_ok)
print("ALL_PNG_DTYPES_OK =", all_png_dtypes_ok)

print(
    "CENTRAL_SHAPE_OK =",
    central.shape == (4, 224, 224),
)

print(
    "CENTRAL_DTYPE_OK =",
    central.dtype == np.uint8,
)

print(
    "CENTRAL_CROP_BITWISE_MATCH =",
    np.array_equal(central_from_png, central),
)

print("DIFF_COUNT =", diff_count)
print("MAX_ABS_DIFF =", max_abs_diff)

=== ESISTENZA FILE ===
ALL_PNG_PATHS_EXIST = True
TENSOR_PATH_EXISTS  = True

=== PNG ORIGINALI ===
channel 0: mode = L size = (272, 224) shape = (224, 272) dtype = uint8
channel 1: mode = L size = (272, 224) shape = (224, 272) dtype = uint8
channel 2: mode = L size = (272, 224) shape = (224, 272) dtype = uint8
channel 3: mode = L size = (272, 224) shape = (224, 272) dtype = uint8

=== TENSORE CENTRALE ===
shape = (4, 224, 224)
dtype = uint8

=== CONTROLLI ===
ALL_PNG_SHAPES_OK = True
ALL_PNG_DTYPES_OK = True
CENTRAL_SHAPE_OK = True
CENTRAL_DTYPE_OK = True
CENTRAL_CROP_BITWISE_MATCH = True
DIFF_COUNT = 0
MAX_ABS_DIFF = 0


In [11]:
import torch

# Replica esplicita della semantica C:
# uint8 -> float32 -> divisione per 255.0f
central_float = (
    central.astype(np.float32)
    / np.float32(255.0)
)

central_torch = torch.from_numpy(
    central_float.copy()
)

print("=== NORMALIZZAZIONE ===")
print("NumPy shape =", central_float.shape)
print("NumPy dtype =", central_float.dtype)
print("Min =", float(central_float.min()))
print("Max =", float(central_float.max()))

print()
print("Torch shape =", tuple(central_torch.shape))
print("Torch dtype =", central_torch.dtype)

print()
print(
    "SHAPE_OK =",
    central_float.shape == (4, 224, 224),
)

print(
    "FLOAT32_OK =",
    central_float.dtype == np.float32,
)

print(
    "RANGE_OK =",
    central_float.min() >= 0.0
    and central_float.max() <= 1.0,
)

# Verifica che la conversione non perda l'informazione uint8 originale.
roundtrip = np.rint(
    central_float * np.float32(255.0)
).astype(np.uint8)

print(
    "UINT8_ROUNDTRIP_OK =",
    np.array_equal(roundtrip, central),
)

print(
    "NUMPY_TORCH_MATCH =",
    np.array_equal(
        central_torch.numpy(),
        central_float,
    ),
)

=== NORMALIZZAZIONE ===
NumPy shape = (4, 224, 224)
NumPy dtype = float32
Min = 0.11764705926179886
Max = 0.843137264251709

Torch shape = (4, 224, 224)
Torch dtype = torch.float32

SHAPE_OK = True
FLOAT32_OK = True
RANGE_OK = True
UINT8_ROUNDTRIP_OK = True
NUMPY_TORCH_MATCH = True


In [12]:
# Primo evento del TRAIN ufficiale
with (METADATA_DIR / "split_train_indices_seed123.csv").open(
    "r", encoding="utf-8", newline=""
) as f:
    train_rows = list(csv.DictReader(f))

train_row = train_rows[0]
train_manifest_index = int(train_row["manifest_index"])
train_event_row = event_rows[train_manifest_index]

train_png_paths = [
    DATA_DIR / train_event_row[column]
    for column in view_columns
]

# Lettura delle 4 viste 272x224
train_png_arrays = []

for path in train_png_paths:
    with Image.open(path) as image:
        train_png_arrays.append(
            np.asarray(image, dtype=np.uint8).copy()
        )

print("=== TRAIN EVENT ===")
print("manifest_index =", train_manifest_index)
print("event_id       =", train_event_row["event_id"])
print(
    "PNG_SHAPES_OK  =",
    all(x.shape == (224, 272) for x in train_png_arrays),
)

# Tre crop di controllo:
# 0  = shift -24
# 24 = centrale
# 48 = shift +24
test_crop_lefts = [0, 24, 48]

crops = {}

for crop_left in test_crop_lefts:
    crop = np.stack(
        [
            image[:, crop_left:crop_left + 224]
            for image in train_png_arrays
        ],
        axis=0,
    )

    crop_float = crop.astype(np.float32) / np.float32(255.0)

    crops[crop_left] = crop

    print()
    print("crop_left =", crop_left)
    print("shape     =", crop.shape)
    print("dtype     =", crop.dtype)
    print("float min =", float(crop_float.min()))
    print("float max =", float(crop_float.max()))
    print(
        "SHAPE_OK  =",
        crop.shape == (4, 224, 224),
    )
    print(
        "RANGE_OK  =",
        crop_float.min() >= 0.0
        and crop_float.max() <= 1.0,
    )

print()
print("=== DIFFERENZE TRA CROP ===")
print(
    "LEFT_VS_CENTER_DIFFERENT =",
    not np.array_equal(crops[0], crops[24]),
)
print(
    "RIGHT_VS_CENTER_DIFFERENT =",
    not np.array_equal(crops[48], crops[24]),
)

=== TRAIN EVENT ===
manifest_index = 7663
event_id       = H1_BqniZ6oLFl
PNG_SHAPES_OK  = True

crop_left = 0
shape     = (4, 224, 224)
dtype     = uint8
float min = 0.03921568766236305
float max = 0.843137264251709
SHAPE_OK  = True
RANGE_OK  = True

crop_left = 24
shape     = (4, 224, 224)
dtype     = uint8
float min = 0.11764705926179886
float max = 0.843137264251709
SHAPE_OK  = True
RANGE_OK  = True

crop_left = 48
shape     = (4, 224, 224)
dtype     = uint8
float min = 0.03921568766236305
float max = 0.843137264251709
SHAPE_OK  = True
RANGE_OK  = True

=== DIFFERENZE TRA CROP ===
LEFT_VS_CENTER_DIFFERENT = True
RIGHT_VS_CENTER_DIFFERENT = True


In [14]:
# Superseded intermediate augmentation check.
# The validated implementation is provided in the following cell.


In [15]:
# Superseded intermediate augmentation check.
# The validated implementation is provided in the following cell.


In [16]:
# ============================================================
# Replica Python dell'augmentation stateless C
# + verifica dei golden vectors ufficiali C
# ============================================================

MASK64 = (1 << 64) - 1

GS_AUGMENT_MODE_LEGACY_TEXT = 0
GS_AUGMENT_MODE_SYMMETRIC_FIGURE = 1

GS_AUGMENT_STATELESS_V1_DOMAIN_TAG = 0x4753415547563121


def c_u64(value):
    """Emula il wrap uint64_t del C."""
    return value & MASK64


def gs_augment_stateless_mix64(value):
    """Replica gs_augment_stateless_mix64() del C."""
    value = c_u64(
        value + 0x9E3779B97F4A7C15
    )

    value = c_u64(
        (value ^ (value >> 30))
        * 0xBF58476D1CE4E5B9
    )

    value = c_u64(
        (value ^ (value >> 27))
        * 0x94D049BB133111EB
    )

    return c_u64(
        value ^ (value >> 31)
    )


def gs_augment_stateless_value(
    base_seed,
    epoch,
    manifest_index,
    mode,
    stream_index,
):
    h = gs_augment_stateless_mix64(
        GS_AUGMENT_STATELESS_V1_DOMAIN_TAG
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(base_seed)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(epoch)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(manifest_index)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(mode)
    )

    return gs_augment_stateless_mix64(
        h ^ c_u64(stream_index)
    )


def gs_augment_crop_left_for_event_mode(
    base_seed,
    epoch,
    manifest_index,
    mode,
):
    if mode == GS_AUGMENT_MODE_LEGACY_TEXT:
        min_crop_left = 0
        max_crop_left = 24

    elif mode == GS_AUGMENT_MODE_SYMMETRIC_FIGURE:
        min_crop_left = 0
        max_crop_left = 48

    else:
        raise ValueError("augmentation mode non valido")

    crop_range = (
        max_crop_left
        - min_crop_left
        + 1
    )

    # Replica esatta di:
    # threshold = (UINT64_C(0) - range) % range;
    threshold = c_u64(
        0 - crop_range
    ) % crop_range

    stream_index = 0

    while True:
        random_value = gs_augment_stateless_value(
            base_seed,
            epoch,
            manifest_index,
            mode,
            stream_index,
        )

        if random_value >= threshold:
            break

        if stream_index == MASK64:
            raise RuntimeError("stream_index overflow")

        stream_index += 1

    crop_left = (
        min_crop_left
        + random_value % crop_range
    )

    return crop_left


# ============================================================
# Golden vectors presi dal test C reale
# ============================================================

golden_vectors = [
    (0, 0, 0,
     GS_AUGMENT_MODE_LEGACY_TEXT, 7),

    (123, 0, 7663,
     GS_AUGMENT_MODE_SYMMETRIC_FIGURE, 36),

    (789, 79, 4224,
     GS_AUGMENT_MODE_SYMMETRIC_FIGURE, 25),

    (456, 3, 42,
     GS_AUGMENT_MODE_LEGACY_TEXT, 0),

    (MASK64, MASK64, MASK64,
     GS_AUGMENT_MODE_SYMMETRIC_FIGURE, 40),
]


print("=== C ↔ PYTHON GOLDEN VECTORS ===")

all_golden_ok = True

for (
    base_seed,
    epoch,
    manifest_index,
    mode,
    expected,
) in golden_vectors:

    actual = gs_augment_crop_left_for_event_mode(
        base_seed,
        epoch,
        manifest_index,
        mode,
    )

    ok = actual == expected

    print(
        "seed =", base_seed,
        "epoch =", epoch,
        "manifest =", manifest_index,
        "mode =", mode,
        "expected =", expected,
        "actual =", actual,
        "OK =", ok,
    )

    if not ok:
        all_golden_ok = False


print()
print(
    "ALL_C_GOLDEN_VECTORS_MATCH =",
    all_golden_ok,
)

=== C ↔ PYTHON GOLDEN VECTORS ===
seed = 0 epoch = 0 manifest = 0 mode = 0 expected = 7 actual = 7 OK = True
seed = 123 epoch = 0 manifest = 7663 mode = 1 expected = 36 actual = 36 OK = True
seed = 789 epoch = 79 manifest = 4224 mode = 1 expected = 25 actual = 25 OK = True
seed = 456 epoch = 3 manifest = 42 mode = 0 expected = 0 actual = 0 OK = True
seed = 18446744073709551615 epoch = 18446744073709551615 manifest = 18446744073709551615 mode = 1 expected = 40 actual = 40 OK = True

ALL_C_GOLDEN_VECTORS_MATCH = True


In [17]:
train_manifest_indices = [
    int(row["manifest_index"])
    for row in train_rows
]

AUGMENTATION_SEED = 789
MODE = GS_AUGMENT_MODE_SYMMETRIC_FIGURE


def crop_map(indices, epoch):
    return {
        manifest_index:
        gs_augment_crop_left_for_event_mode(
            AUGMENTATION_SEED,
            epoch,
            manifest_index,
            MODE,
        )
        for manifest_index in indices
    }


# Ordine originale
map_original = crop_map(
    train_manifest_indices,
    epoch=0,
)

# Ordine invertito
map_reversed = crop_map(
    reversed(train_manifest_indices),
    epoch=0,
)

# Simulazione di confini di batch diversi
map_chunked = {}

batch_size_test = 37

for start in range(
    0,
    len(train_manifest_indices),
    batch_size_test,
):
    chunk = train_manifest_indices[
        start:start + batch_size_test
    ]

    map_chunked.update(
        crop_map(chunk, epoch=0)
    )


# Epoch successiva
map_epoch1 = crop_map(
    train_manifest_indices,
    epoch=1,
)

changed_between_epochs = sum(
    map_original[index] != map_epoch1[index]
    for index in train_manifest_indices
)

values_epoch0 = list(map_original.values())


print("=== FULL TRAIN STATELESS AUDIT ===")

print(
    "ORDER_INDEPENDENT =",
    map_original == map_reversed,
)

print(
    "BATCH_BOUNDARY_INDEPENDENT =",
    map_original == map_chunked,
)

print(
    "ALL_IN_0_48 =",
    all(
        0 <= value <= 48
        for value in values_epoch0
    ),
)

print(
    "DISTINCT_CROPS_EPOCH0 =",
    len(set(values_epoch0)),
)

print(
    "MIN_CROP_EPOCH0 =",
    min(values_epoch0),
)

print(
    "MAX_CROP_EPOCH0 =",
    max(values_epoch0),
)

print(
    "EVENTS_CHANGED_EPOCH0_TO_1 =",
    changed_between_epochs,
)

print(
    "FIRST_TRAIN_MANIFEST_INDEX =",
    train_manifest_indices[0],
)

print(
    "FIRST_TRAIN_CROP_EPOCH0 =",
    map_original[train_manifest_indices[0]],
)

print(
    "FIRST_TRAIN_CROP_EPOCH1 =",
    map_epoch1[train_manifest_indices[0]],
)

=== FULL TRAIN STATELESS AUDIT ===
ORDER_INDEPENDENT = True
BATCH_BOUNDARY_INDEPENDENT = True
ALL_IN_0_48 = True
DISTINCT_CROPS_EPOCH0 = 49
MIN_CROP_EPOCH0 = 0
MAX_CROP_EPOCH0 = 48
EVENTS_CHANGED_EPOCH0_TO_1 = 6715
FIRST_TRAIN_MANIFEST_INDEX = 7663
FIRST_TRAIN_CROP_EPOCH0 = 36
FIRST_TRAIN_CROP_EPOCH1 = 48


In [18]:
def load_event_crop_float01(event_row, crop_left):
    """
    Replica Python della semantica di gs_event_load_crop_float01().
    """

    columns = [
        "img_0_5_272",
        "img_1_0_272",
        "img_2_0_272",
        "img_4_0_272",
    ]

    channels = []

    for column in columns:
        path = DATA_DIR / event_row[column]

        with Image.open(path) as image:
            if image.mode != "L":
                raise RuntimeError(
                    f"PNG non monocanale: {path}"
                )

            array = np.asarray(
                image,
                dtype=np.uint8,
            )

        if array.shape != (224, 272):
            raise RuntimeError(
                f"Shape PNG non valida: {array.shape}"
            )

        crop = array[
            :,
            crop_left:crop_left + 224,
        ]

        if crop.shape != (224, 224):
            raise RuntimeError(
                f"Crop non valido: {crop.shape}"
            )

        channels.append(crop)

    tensor_u8 = np.stack(
        channels,
        axis=0,
    )

    tensor_f32 = (
        tensor_u8.astype(np.float32)
        / np.float32(255.0)
    )

    return tensor_u8, tensor_f32


# Primo evento train ufficiale validato in precedenza
manifest_index = train_manifest_indices[0]
event = event_rows[manifest_index]

crop_epoch0 = gs_augment_crop_left_for_event_mode(
    789,
    0,
    manifest_index,
    GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
)

crop_epoch1 = gs_augment_crop_left_for_event_mode(
    789,
    1,
    manifest_index,
    GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
)

u8_epoch0, f32_epoch0 = load_event_crop_float01(
    event,
    crop_epoch0,
)

u8_epoch1, f32_epoch1 = load_event_crop_float01(
    event,
    crop_epoch1,
)

print("=== TRAIN RNG → DATA AUDIT ===")
print("manifest_index =", manifest_index)
print("event_id       =", event["event_id"])

print()
print("crop epoch 0 =", crop_epoch0)
print("crop epoch 1 =", crop_epoch1)

print()
print("u8 shape      =", u8_epoch0.shape)
print("u8 dtype      =", u8_epoch0.dtype)
print("f32 shape     =", f32_epoch0.shape)
print("f32 dtype     =", f32_epoch0.dtype)

print()
print(
    "SHAPE_OK =",
    u8_epoch0.shape == (4, 224, 224),
)

print(
    "FLOAT32_OK =",
    f32_epoch0.dtype == np.float32,
)

print(
    "RANGE_OK =",
    f32_epoch0.min() >= 0.0
    and f32_epoch0.max() <= 1.0,
)

print(
    "EPOCH0_VS_EPOCH1_DIFFERENT =",
    not np.array_equal(
        u8_epoch0,
        u8_epoch1,
    ),
)

# Controllo esplicito contro il crop già letto manualmente.
manual_epoch0 = np.stack(
    [
        image[
            :,
            crop_epoch0:crop_epoch0 + 224
        ]
        for image in train_png_arrays
    ],
    axis=0,
)

print(
    "EXPLICIT_CROP_MATCH =",
    np.array_equal(
        u8_epoch0,
        manual_epoch0,
    ),
)

roundtrip = np.rint(
    f32_epoch0 * np.float32(255.0)
).astype(np.uint8)

print(
    "UINT8_ROUNDTRIP_OK =",
    np.array_equal(
        roundtrip,
        u8_epoch0,
    ),
)

=== TRAIN RNG → DATA AUDIT ===
manifest_index = 7663
event_id       = H1_BqniZ6oLFl

crop epoch 0 = 36
crop epoch 1 = 48

u8 shape      = (4, 224, 224)
u8 dtype      = uint8
f32 shape     = (4, 224, 224)
f32 dtype     = float32

SHAPE_OK = True
FLOAT32_OK = True
RANGE_OK = True
EPOCH0_VS_EPOCH1_DIFFERENT = True
EXPLICIT_CROP_MATCH = True
UINT8_ROUNDTRIP_OK = True


In [19]:
print(
    "DATA_DIR_EXISTS =",
    (LOCAL_RUNTIME_ROOT / "sakai224_full_data").is_dir(),
)

print(
    "PROJECT_ROOT_EXISTS =",
    PROJECT_ROOT.is_dir(),
)


DATA_DIR_EXISTS = True
DRIVE_MOUNT_EXISTS = True


In [20]:
# Superseded intermediate data-audit attempt.
# The validated implementation is provided in the subsequent audit cells.


In [21]:
# Superseded intermediate data-audit attempt.
# The validated implementation is provided in the subsequent audit cells.


In [22]:
from pathlib import Path
from PIL import Image
import csv
import numpy as np

METADATA_DIR = LOCAL_RUNTIME_ROOT / "sakai224_full_transfer_v1"
DATA_DIR = LOCAL_RUNTIME_ROOT / "sakai224_full_data"

EVENT_MANIFEST = METADATA_DIR / "manifest_events_272.csv"
TRAIN_SPLIT = METADATA_DIR / "split_train_indices_seed123.csv"


def read_csv_dicts(path):
    with open(
        path,
        "r",
        encoding="utf-8",
        newline=""
    ) as file:
        return list(csv.DictReader(file))


event_rows = read_csv_dicts(EVENT_MANIFEST)
train_rows = read_csv_dicts(TRAIN_SPLIT)

print("EVENT_ROWS =", len(event_rows))
print("TRAIN_ROWS =", len(train_rows))
print("METADATA_OK =", len(event_rows) == 8583 and len(train_rows) == 6866)

EVENT_ROWS = 8583
TRAIN_ROWS = 6866
METADATA_OK = True


In [23]:
MASK64 = (1 << 64) - 1

GS_AUGMENT_MODE_LEGACY_TEXT = 0
GS_AUGMENT_MODE_SYMMETRIC_FIGURE = 1

GS_AUGMENT_STATELESS_V1_DOMAIN_TAG = 0x4753415547563121


def c_u64(value):
    return value & MASK64


def gs_augment_stateless_mix64(value):
    value = c_u64(value + 0x9E3779B97F4A7C15)

    value = c_u64(
        (value ^ (value >> 30))
        * 0xBF58476D1CE4E5B9
    )

    value = c_u64(
        (value ^ (value >> 27))
        * 0x94D049BB133111EB
    )

    return c_u64(value ^ (value >> 31))


def gs_augment_stateless_value(
    base_seed,
    epoch,
    manifest_index,
    mode,
    stream_index,
):
    h = gs_augment_stateless_mix64(
        GS_AUGMENT_STATELESS_V1_DOMAIN_TAG
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(base_seed)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(epoch)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(manifest_index)
    )

    h = gs_augment_stateless_mix64(
        h ^ c_u64(mode)
    )

    return gs_augment_stateless_mix64(
        h ^ c_u64(stream_index)
    )


def gs_augment_crop_left_for_event_mode(
    base_seed,
    epoch,
    manifest_index,
    mode,
):
    if mode == GS_AUGMENT_MODE_LEGACY_TEXT:
        min_crop_left = 0
        max_crop_left = 24

    elif mode == GS_AUGMENT_MODE_SYMMETRIC_FIGURE:
        min_crop_left = 0
        max_crop_left = 48

    else:
        raise ValueError("mode non valido")

    crop_range = max_crop_left - min_crop_left + 1

    threshold = c_u64(
        0 - crop_range
    ) % crop_range

    stream_index = 0

    while True:
        random_value = gs_augment_stateless_value(
            base_seed,
            epoch,
            manifest_index,
            mode,
            stream_index,
        )

        if random_value >= threshold:
            return (
                min_crop_left
                + random_value % crop_range
            )

        stream_index += 1


# Known-answer C
golden_vectors = [
    (0, 0, 0, 0, 7),
    (123, 0, 7663, 1, 36),
    (789, 79, 4224, 1, 25),
    (456, 3, 42, 0, 0),
    (MASK64, MASK64, MASK64, 1, 40),
]

golden_ok = all(
    gs_augment_crop_left_for_event_mode(
        seed,
        epoch,
        manifest_index,
        mode,
    ) == expected
    for seed, epoch, manifest_index, mode, expected
    in golden_vectors
)

first_manifest_index = int(
    train_rows[0]["manifest_index"]
)

print(
    "GOLDEN_VECTORS_OK =",
    golden_ok,
)

print(
    "FIRST_MANIFEST_INDEX =",
    first_manifest_index,
)

print(
    "CROP_EPOCH0 =",
    gs_augment_crop_left_for_event_mode(
        789,
        0,
        first_manifest_index,
        GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
    ),
)

print(
    "CROP_EPOCH1 =",
    gs_augment_crop_left_for_event_mode(
        789,
        1,
        first_manifest_index,
        GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
    ),
)

GOLDEN_VECTORS_OK = True
FIRST_MANIFEST_INDEX = 7663
CROP_EPOCH0 = 36
CROP_EPOCH1 = 48


In [24]:
VIEW_COLUMNS = [
    "img_0_5_272",
    "img_1_0_272",
    "img_2_0_272",
    "img_4_0_272",
]

manifest_index = first_manifest_index
event = event_rows[manifest_index]

crop0 = gs_augment_crop_left_for_event_mode(
    789,
    0,
    manifest_index,
    GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
)

crop1 = gs_augment_crop_left_for_event_mode(
    789,
    1,
    manifest_index,
    GS_AUGMENT_MODE_SYMMETRIC_FIGURE,
)


def load_crop_u8(event_row, crop_left):
    channels = []

    for column in VIEW_COLUMNS:
        path = DATA_DIR / event_row[column]

        with Image.open(path) as image:
            if image.mode != "L":
                raise RuntimeError(
                    f"PNG non grayscale: {path}"
                )

            array = np.asarray(
                image,
                dtype=np.uint8,
            ).copy()

        if array.shape != (224, 272):
            raise RuntimeError(
                f"Shape errata: {array.shape}"
            )

        channels.append(
            array[:, crop_left:crop_left + 224]
        )

    return np.stack(channels, axis=0)


u8_epoch0 = load_crop_u8(event, crop0)
u8_epoch1 = load_crop_u8(event, crop1)

f32_epoch0 = (
    u8_epoch0.astype(np.float32)
    / np.float32(255.0)
)

roundtrip = np.rint(
    f32_epoch0 * np.float32(255.0)
).astype(np.uint8)


checks = {
    "MANIFEST_INDEX_OK":
        manifest_index == 7663,

    "EVENT_ID_OK":
        event["event_id"] == "H1_BqniZ6oLFl",

    "CROP_EPOCH0_OK":
        crop0 == 36,

    "CROP_EPOCH1_OK":
        crop1 == 48,

    "SHAPE_OK":
        u8_epoch0.shape == (4, 224, 224),

    "UINT8_OK":
        u8_epoch0.dtype == np.uint8,

    "FLOAT32_OK":
        f32_epoch0.dtype == np.float32,

    "RANGE_OK":
        (
            f32_epoch0.min() >= 0.0
            and f32_epoch0.max() <= 1.0
        ),

    "EPOCH0_VS_EPOCH1_DIFFERENT":
        not np.array_equal(
            u8_epoch0,
            u8_epoch1,
        ),

    "UINT8_ROUNDTRIP_OK":
        np.array_equal(
            roundtrip,
            u8_epoch0,
        ),
}


print("=== FRESH RNG -> REAL DATA AUDIT ===")
print("manifest_index =", manifest_index)
print("event_id       =", event["event_id"])
print("crop epoch 0   =", crop0)
print("crop epoch 1   =", crop1)
print("shape          =", u8_epoch0.shape)
print("float min      =", float(f32_epoch0.min()))
print("float max      =", float(f32_epoch0.max()))
print()

for name, value in checks.items():
    print(name, "=", value)

print()
print(
    "FRESH_RNG_DATA_AUDIT_PASS =",
    all(checks.values()),
)

=== FRESH RNG -> REAL DATA AUDIT ===
manifest_index = 7663
event_id       = H1_BqniZ6oLFl
crop epoch 0   = 36
crop epoch 1   = 48
shape          = (4, 224, 224)
float min      = 0.11764705926179886
float max      = 0.843137264251709

MANIFEST_INDEX_OK = True
EVENT_ID_OK = True
CROP_EPOCH0_OK = True
CROP_EPOCH1_OK = True
SHAPE_OK = True
UINT8_OK = True
FLOAT32_OK = True
RANGE_OK = True
EPOCH0_VS_EPOCH1_DIFFERENT = True
UINT8_ROUNDTRIP_OK = True

FRESH_RNG_DATA_AUDIT_PASS = True


In [ ]:
if IN_COLAB and not PROJECT_ROOT_IS_EXPLICIT:
    from google.colab import drive

    if not COLAB_MYDRIVE_ROOT.is_dir():
        drive.mount(str(COLAB_DRIVE_MOUNT_POINT), force_remount=False)
    else:
        print("Drive già disponibile:", COLAB_DRIVE_MOUNT_POINT)
else:
    print("Recovery mount non richiesto; PROJECT_ROOT =", PROJECT_ROOT)


Mounted at /content/drive


In [ ]:
print("PROJECT_ROOT_OK =", PROJECT_ROOT.is_dir())
print("PROJECT_ROOT =", PROJECT_ROOT)


DRIVE_MOUNT_OK = True
MOUNT_PATH = /content/drive/MyDrive
